<a href="https://colab.research.google.com/github/gyxcit/hyperclap/blob/main/clap_music_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 LAION CLAP + Whisper — Analyse Musicale Complète

**Ce notebook permet de :**
- Tagger automatiquement une chanson (genre, émotion, instruments, tempo...)
- Transcrire les lyrics avec timestamps
- Détecter automatiquement couplets et refrains
- Télécharger les résultats en JSON, TXT, CSV et ZIP

> ⚡ **Activer le GPU** : Exécution → Modifier le type d'exécution → GPU (T4)

In [1]:
# CELLULE 1 : Installation
!pip install laion-clap librosa soundfile scikit-learn --quiet
!pip install openai-whisper --quiet

In [2]:
# CELLULE 2 : Patch PyTorch 2.6 + Chargement CLAP
import torch, functools
torch.load = functools.partial(torch.load, weights_only=False)

import laion_clap
clap_model = laion_clap.CLAP_Module(enable_fusion=False, amodel='HTSAT-tiny')
clap_model.load_ckpt()
clap_model.eval()
print('Modele CLAP charge !')

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load our best checkpoint in the paper.
Download completed!
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.weight 	 Loaded
aud

In [3]:
# CELLULE 3 : Chargement Whisper large-v3
import whisper
whisper_model = whisper.load_model('large-v3')
print('Modele Whisper large-v3 charge !')

100%|██████████████████████████████████████| 2.88G/2.88G [00:22<00:00, 136MiB/s]


Modele Whisper large-v3 charge !


In [4]:
# CELLULE 4 : Categories CLAP
CATEGORIES = {
    'Genre musical': [
        'rock music', 'pop music', 'jazz music', 'classical music',
        'hip hop music', 'electronic music', 'reggae music', 'metal music',
        'blues music', 'folk music', 'R&B music', 'country music',
    ],
    'Rap & Hip-Hop': [
        'French rap music', 'old school hip hop', 'trap music',
        'rap with aggressive lyrics', 'rap with melodic singing',
        'rap with heavy bass and 808s', 'rap with jazzy beats',
        'rap with orchestral strings', 'rap with drill beats',
        'rap freestyle music', 'rap with soulful samples', 'gangsta rap music',
    ],
    'Musique Francaise': [
        'French chanson music', 'French variete music', 'French pop music',
        'French electro music', 'French jazz manouche',
        'French Caribbean zouk music', 'French Antillean music',
        'French rap urbain', 'French romantic ballad',
        'French cabaret music', 'French accordion music', 'French new wave music',
    ],
    'Emotion': [
        'happy and joyful music', 'sad and melancholic music',
        'energetic and exciting music', 'calm and relaxing music',
        'romantic music', 'dark and tense music',
        'uplifting and inspiring music', 'nostalgic music',
        'angry and rebellious music', 'melancholic and introspective music',
    ],
    'Instruments': [
        'music with electric guitar', 'music with acoustic guitar',
        'music with piano', 'music with drums and percussion',
        'music with bass guitar', 'music with violin or strings',
        'music with synthesizer', 'music with trumpet or brass',
        'music with saxophone', 'music with vocals and singing',
        'music with turntables and scratching', 'music with accordion',
    ],
    'Tempo et Ambiance': [
        'fast tempo music', 'slow tempo music', 'dance music',
        'background ambient music', 'aggressive and intense music',
        'smooth and groovy music', 'acoustic and unplugged music',
        'loud and powerful music', 'lofi chill music',
        'street music with urban vibe',
    ],
    'Influence et Origine': [
        'African influenced music', 'Caribbean influenced music',
        'American influenced music', 'Latin influenced music',
        'Arabic influenced music', 'music with African percussion',
        'music with Latin rhythm', 'music blending French and African styles',
    ],
    'Usage et Contexte': [
        'party music', 'workout and gym music', 'study and focus music',
        'late night driving music', 'summer music', 'street and urban music',
        'club and nightlife music', 'music for relaxation',
    ],
}

CUSTOM_CATEGORIES = {}
ALL_CATEGORIES = {**CATEGORIES, **CUSTOM_CATEGORIES}
print(f'{len(ALL_CATEGORIES)} categories chargees')

8 categories chargees


In [5]:
# CELLULE 5 : Fonctions CLAP Tagging
import librosa
import numpy as np
from sklearn.preprocessing import normalize


def tag_audio(audio_path, model, categories, threshold=0.08):
    audio, _ = librosa.load(audio_path, sr=48000, mono=True)
    max_samples = 48000 * 10
    if len(audio) > max_samples:
        start = (len(audio) - max_samples) // 2
        audio = audio[start:start + max_samples]
    audio_embed = model.get_audio_embedding_from_data(x=audio.reshape(1, -1), use_tensor=False)
    audio_norm = normalize(audio_embed)
    results = {}
    for cat_name, labels in categories.items():
        text_embed = model.get_text_embedding(labels, use_tensor=False)
        text_norm = normalize(text_embed)
        scores = (audio_norm @ text_norm.T)[0]
        kept = [(labels[i], float(scores[i])) for i in scores.argsort()[::-1] if scores[i] >= threshold]
        results[cat_name] = kept
    return results


def print_tags(results):
    print('\n' + '=' * 58)
    print('  TAGS MUSICAUX')
    print('=' * 58)
    total = 0
    for cat_name, tags in results.items():
        if not tags:
            continue
        print(f"\n{cat_name}  ({len(tags)} tags)")
        print('-' * 48)
        for label, score in tags:
            bar = '#' * int(score * 100)
            print(f'  {label:40s} {score:.3f}  {bar}')
        total += len(tags)
    print(f'\nTotal : {total} tags detectes')
    print('=' * 58)


print('Fonctions CLAP pretes')

Fonctions CLAP pretes


In [6]:
# CELLULE 6 : Fonctions Whisper Lyrics

def format_timestamp(seconds):
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f'{m:02d}:{s:02d}'


def detect_structure(segments):
    from difflib import SequenceMatcher
    def similarity(a, b):
        return SequenceMatcher(None, a.lower(), b.lower()).ratio()
    texts = [s['text'].strip() for s in segments]
    n = len(texts)
    sim_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                sim_matrix[i][j] = similarity(texts[i], texts[j])
    REFRAIN_THRESHOLD = 0.55
    section_map = {}
    refrain_groups = []
    couplet_count = 0
    for i in range(n):
        if i in section_map:
            continue
        max_sim = sim_matrix[i].max()
        if max_sim >= REFRAIN_THRESHOLD:
            similar_indices = [j for j in range(n) if sim_matrix[i][j] >= REFRAIN_THRESHOLD]
            similar_indices.append(i)
            merged = False
            for group in refrain_groups:
                if any(idx in group for idx in similar_indices):
                    group.update(similar_indices)
                    merged = True
                    break
            if not merged:
                refrain_groups.append(set(similar_indices))
        else:
            couplet_count += 1
            section_map[i] = f'Couplet {couplet_count}'
    for r_idx, group in enumerate(refrain_groups):
        label = 'Refrain' if len(refrain_groups) == 1 else f'Refrain {r_idx+1}'
        for idx in group:
            section_map[idx] = label
    for i in range(n):
        if i not in section_map:
            couplet_count += 1
            section_map[i] = f'Couplet {couplet_count}'
    return [{**seg, 'section': section_map[i]} for i, seg in enumerate(segments)]


def transcribe_lyrics(audio_path, whisper_model, language='fr'):
    print(f'Transcription : {audio_path}')
    print('(1-3 min selon la duree...)\n')
    result = whisper_model.transcribe(
        audio_path,
        language=language,
        word_timestamps=True,
        verbose=False,
        condition_on_previous_text=True,
        compression_ratio_threshold=2.4,
        no_speech_threshold=0.6,
        initial_prompt='Transcription de rap et argot francais. Vocabulaire urbain : wesh, ouf, daron, meuf.',
    )
    grouped = []
    buffer_text = ''
    buffer_start = None
    buffer_end = None
    GROUP_DURATION = 4.0
    for seg in result['segments']:
        if buffer_start is None:
            buffer_start = seg['start']
        buffer_text += ' ' + seg['text'].strip()
        buffer_end = seg['end']
        if (buffer_end - buffer_start) >= GROUP_DURATION:
            grouped.append({'start': buffer_start, 'end': buffer_end, 'text': buffer_text.strip()})
            buffer_text = ''
            buffer_start = None
    if buffer_text.strip():
        grouped.append({'start': buffer_start, 'end': buffer_end, 'text': buffer_text.strip()})
    structured = detect_structure(grouped)
    return {
        'full_text': result['text'],
        'language': result.get('language', language or '?'),
        'segments': grouped,
        'structured': structured,
    }


def print_lyrics(lyrics_result):
    print('\n' + '=' * 58)
    print('  LYRICS - TRANSCRIPTION')
    print(f"  Langue : {lyrics_result['language'].upper()}")
    print('=' * 58)
    current_section = None
    for seg in lyrics_result['structured']:
        section = seg['section']
        ts = f"[{format_timestamp(seg['start'])} -> {format_timestamp(seg['end'])}]"
        if section != current_section:
            print(f'\n  -- {section} --')
            current_section = section
        print(f'  {ts}  {seg["text"]}')
    print('\n' + '=' * 58)
    print('  TEXTE COMPLET')
    print('=' * 58)
    print(lyrics_result['full_text'])
    print('=' * 58)


print('Fonctions Whisper pretes')

Fonctions Whisper pretes


In [7]:
# CELLULE 7 : Fonctions Export et Telechargement
import json
import csv
import zipfile
import os
from google.colab import files


def export_json(audio_path, tags, lyrics):
    base = os.path.splitext(os.path.basename(audio_path))[0]
    output_path = f'/content/{base}_analysis.json'
    data = {
        'file': audio_path,
        'tags': {cat: [{'label': l, 'score': round(s, 4)} for l, s in items] for cat, items in tags.items() if items},
        'lyrics': {
            'language': lyrics['language'],
            'full_text': lyrics['full_text'],
            'structured': [{'section': seg['section'], 'start': round(seg['start'], 2), 'end': round(seg['end'], 2), 'text': seg['text']} for seg in lyrics['structured']],
        },
    }
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f'  OK JSON  -> {output_path}')
    return output_path


def export_txt(audio_path, tags, lyrics):
    base = os.path.splitext(os.path.basename(audio_path))[0]
    output_path = f'/content/{base}_analysis.txt'
    lines = []
    lines.append('=' * 60)
    lines.append(f'  ANALYSE MUSICALE - {os.path.basename(audio_path)}')
    lines.append('=' * 60)
    lines.append('')
    lines.append('  TAGS MUSICAUX')
    lines.append('-' * 60)
    for cat_name, tag_list in tags.items():
        if not tag_list:
            continue
        lines.append(f'\n  {cat_name}')
        for label, score in tag_list:
            lines.append(f'    - {label:38s} {score:.3f}')
    lines.append('')
    lines.append('=' * 60)
    lines.append(f"  LYRICS  (langue : {lyrics['language'].upper()})")
    lines.append('=' * 60)
    current_section = None
    for seg in lyrics['structured']:
        section = seg['section']
        if section != current_section:
            lines.append(f'\n  -- {section} --')
            current_section = section
        ts = f"[{format_timestamp(seg['start'])} -> {format_timestamp(seg['end'])}]"
        lines.append(f'  {ts}  {seg["text"]}')
    lines.append('')
    lines.append('=' * 60)
    lines.append('  TEXTE COMPLET')
    lines.append('=' * 60)
    lines.append(lyrics['full_text'])
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f'  OK TXT  -> {output_path}')
    return output_path


def export_csv(audio_path, tags):
    base = os.path.splitext(os.path.basename(audio_path))[0]
    output_path = f'/content/{base}_tags.csv'
    rows = []
    for cat_name, tag_list in tags.items():
        for label, score in tag_list:
            rows.append({'fichier': os.path.basename(audio_path), 'categorie': cat_name, 'label': label, 'score': round(score, 4)})
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['fichier', 'categorie', 'label', 'score'])
        writer.writeheader()
        writer.writerows(rows)
    print(f'  OK CSV  -> {output_path}')
    return output_path


def export_all_and_download(audio_path, tags, lyrics):
    base = os.path.splitext(os.path.basename(audio_path))[0]
    zip_path = f'/content/{base}_results.zip'
    print('\nExport des fichiers...')
    path_json = export_json(audio_path, tags, lyrics)
    path_txt  = export_txt(audio_path, tags, lyrics)
    path_csv  = export_csv(audio_path, tags)
    print(f'\nCompression -> {zip_path}')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(path_json, os.path.basename(path_json))
        zf.write(path_txt,  os.path.basename(path_txt))
        zf.write(path_csv,  os.path.basename(path_csv))
    print('\nTelechargement...')
    files.download(zip_path)
    print(f'\nZIP telecharge : {os.path.basename(zip_path)}')
    return zip_path


print('Fonctions export pretes')

Fonctions export pretes


In [9]:
# CELLULE 8 : ANALYSE PRINCIPALE
# Modifie AUDIO_PATH avec le chemin de ton fichier

AUDIO_PATH = '/content/niska.mp3'   # <- CHANGE ICI
THRESHOLD  = 0.08                   # 0.08 permissif | 0.12 modere | 0.15 strict
LANGUAGE   = 'fr'                   # 'fr' | 'en' | None (auto-detection)

# 1. Tagging CLAP
print('Analyse des tags musicaux (CLAP)...')
tags = tag_audio(AUDIO_PATH, clap_model, ALL_CATEGORIES, threshold=THRESHOLD)
print_tags(tags)

# 2. Transcription Whisper
lyrics = transcribe_lyrics(AUDIO_PATH, whisper_model, language=LANGUAGE)
print_lyrics(lyrics)

Analyse des tags musicaux (CLAP)...

  TAGS MUSICAUX

Genre musical  (11 tags)
------------------------------------------------
  hip hop music                            0.368  ####################################
  R&B music                                0.365  ####################################
  pop music                                0.304  ##############################
  blues music                              0.290  ############################
  reggae music                             0.285  ############################
  country music                            0.248  ########################
  metal music                              0.166  ################
  rock music                               0.166  ################
  electronic music                         0.162  ################
  folk music                               0.113  ###########
  jazz music                               0.111  ###########

Rap & Hip-Hop  (10 tags)
---------------------------------

 94%|█████████▍| 19908/21120 [02:38<00:09, 125.81frames/s]



  LYRICS - TRANSCRIPTION
  Langue : FR

  -- Couplet 1 --
  [00:28 -> 00:34]  Devant dans la bagarre, les autres c'est des bâtards, j'peux compter sur personne, j'kiffe que sur mon gun. Et j'peux pas l'entretenir, je sais qu'c'est une putain,

  -- Couplet 2 --
  [00:34 -> 00:40]  en plus c'est une gratteuse, mais qu'est-ce qu'elle est bonne ? C'est tout pour la mama, tu crois qu'on ment quand j'te dis qu'on a ramé ?

  -- Couplet 3 --
  [00:40 -> 00:44]  Tu dis qu'on a ramé ? On n'est plus des gamins, après les pires c'est les truffes qu'on va damer. Boubouboubou !

  -- Couplet 4 --
  [00:44 -> 00:51]  Et les autres c'est des bords, ils m'ont lancé des sorts, ils m'ont souhaité la mort, mais ça va aller, si ça doit s'allumer, bah ça va s'allumer.

  -- Couplet 5 --
  [00:51 -> 00:56]  Et j'trouve un alibi, puis je me taille, du lundi au lundi, bel-ec y'a les 22, mes potos sont cramés, j'reponds plus au fun,

  -- Couplet 6 --
  [00:56 -> 01:02]  du Gucci au Fendi, rien qu'j'enchaîne

In [10]:
# CELLULE 9 : TELECHARGEMENT DES RESULTATS
# Genere un ZIP contenant JSON + TXT + CSV
export_all_and_download(AUDIO_PATH, tags, lyrics)


Export des fichiers...
  OK JSON  -> /content/niska_analysis.json
  OK TXT  -> /content/niska_analysis.txt
  OK CSV  -> /content/niska_tags.csv

Compression -> /content/niska_results.zip

Telechargement...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


ZIP telecharge : niska_results.zip


'/content/niska_results.zip'

In [11]:
# CELLULE 10 (optionnelle) : Uploader ton propre fichier audio
# from google.colab import files
# uploaded = files.upload()
# AUDIO_PATH = '/content/' + list(uploaded.keys())[0]
# print(f'Fichier uploade : {AUDIO_PATH}')
# Puis relance les cellules 8 et 9

In [12]:
# CELLULE 11 (optionnelle) : Ajouter des categories custom
# CUSTOM_CATEGORIES['Sous-genre Rap'] = [
#     'drill music', 'cloud rap', 'boom bap', 'phonk music'
# ]
# ALL_CATEGORIES = {**CATEGORIES, **CUSTOM_CATEGORIES}
# tags = tag_audio(AUDIO_PATH, clap_model, ALL_CATEGORIES, threshold=THRESHOLD)
# print_tags(tags)